In [1]:
import numpy as np
import xarray as xr
from matplotlib import pyplot as plt
import pandas as pd
import matplotlib.cm as cm
from matplotlib.patches import Rectangle
from fonts_config import set_computer_modern, truncate_colormap
set_computer_modern()
import matplotlib as mpl
mpl.rcParams['axes.unicode_minus'] = False
from matplotlib.patches import Rectangle, Patch
import matplotlib.lines as mlines
import matplotlib.patheffects as path_effects
import colormaps as cmaps
import matplotlib.colors as mcolors
from matplotlib.colors import LightSource, LogNorm, TwoSlopeNorm
import matplotlib.gridspec as gridspec


In [2]:
# ----------------------------------------------------------------------------
# Paths
# ----------------------------------------------------------------------------

rsl=xr.open_dataset("../../FesmData/Gowan2023_GAPSLIP_GrIS/output/rsl_dataset_reduced.nc")
gia=xr.open_dataset("../../FesmData/Schumacher2018_GIA_GrIS/data/schumacher2018_GR.nc")
topo = xr.open_dataset("/p/projects/megarun/ice_data/Greenland/GRL-8KM/GRL-8KM_TOPO-M17.nc")
vel = xr.open_dataset("/p/projects/megarun/ice_data/Greenland/GRL-8KM/GRL-8KM_VEL-J18.nc")
iso=xr.open_dataset("../../FesmData/Leger2024_PaleoGris/output/paleogris_8km.nc")
bui = pd.read_excel("../../FesmData/Buizert2018/grl56971-sup-0002-supinfo.xlsx", header=10)
bui=bui.drop(0)
icec_id=["ngrip","grip","camp_century","dye3"]
icec=np.array([[75.0166666,72.5833,77.1667,65.1833326],[ -42.5333312,-37.6333,-61.1333,-43.8166634]])
rsl=xr.open_dataset("../../FesmData/Gowan2023_GAPSLIP_GrIS/output/rsl_dataset_reduced.nc")
vinter = xr.open_dataset("../../FesmData/Vinther2009_elevations/vinther2009.nc")


In [ ]:

def add_map2(ax, topo, iso, vel):
    lgm=xr.open_dataset("/p/projects/megarun/luciagu/data/leger2024/lgm.nc")
    obs_v = xr.open_dataset("/p/projects/megarun/ice_data/Greenland/GRL-8KM/GRL-8KM_VEL.nc")

    lgm_leg = xr.where((lgm.mask==3)|(lgm.mask==2),1,0)
    log_uxy = obs_v.uxy_srf.where(topo.H_ice>0) + 1e-8
    uxy_s_map = truncate_colormap(plt.get_cmap('gray_r'), minval=0, maxval=0.9)

    ls = LightSource(azdeg=315, altdeg=45)
    zmap = mcolors.LinearSegmentedColormap.from_list("zmap", [
            (0.0, "#5E88AA"),
            (0.499, "#ADC3CF"),
            (0.50, "#024224"),
            (1.0, "#b18b5d")])
    norm = TwoSlopeNorm(vmin=topo.z_bed.min(), vcenter=0, vmax=topo.z_bed.max())
    rgb = ls.shade(topo.z_bed.values, cmap=zmap, norm=norm, blend_mode='soft', vert_exag=0.1)
    bat = ax.pcolormesh(topo.xc, topo.yc, topo.z_bed*1e-3, cmap=zmap, vmin=-3, vmax=3)
    img = ax.imshow(rgb, extent=[-5, 5, -5, 5])
    ax.contourf(topo.xc, topo.yc, topo.H_ice.where(topo.H_ice>10), levels=np.linspace(0,4000,2), colors="#E6F1F3",zorder=3)
    vels = ax.contourf(topo.xc, topo.yc, log_uxy, log_uxy, levels = np.logspace(0,3,10) ,  norm = LogNorm(), cmap = uxy_s_map,extend='both',zorder=3)
    cs1 = ax.contour(topo.xc, topo.yc, topo.z_srf, levels=np.arange(150, 3600, 500), colors='black', alpha=0.3,zorder=4)
    cs2 = ax.contour(topo.xc, topo.yc, topo.z_srf, levels=np.arange(150, 4150, 1000), colors='black',zorder=4)

    # Ice core locations
    for (lat,lon) in zip(icec[0,:], icec[1,:]):
        dist = np.sqrt((topo.lat2D - lat)**2 + (topo.lon2D - lon)**2)
        iy, ix = np.unravel_index(dist.argmin(), dist.shape)
        x = topo.xc[ix]
        y = topo.yc[iy]
        ax.plot(x, y, '*', color="black", markersize=10, markeredgecolor='black', zorder=6)  
    ax.text(-450,-1290,"Camp Century", fontsize=12,bbox=dict(facecolor='white', alpha=0.6, edgecolor='none'), zorder=6)  
    ax.text(0,-1595,"NGRIP", fontsize=12,bbox=dict(facecolor='white', alpha=0.6, edgecolor='none'), zorder=6)  
    ax.text(150,-1830,"GRIP", fontsize=12,bbox=dict(facecolor='white', alpha=0.6, edgecolor='none'), zorder=6)  
    ax.text(0,-2670,"DYE-3", fontsize=12,bbox=dict(facecolor='white', alpha=0.6, edgecolor='none'), zorder=6)  
    
    # Isochrones
    lim=14
    cmap_name =	cmaps.amwg_r
    cmap = cm.get_cmap(cmap_name, lim)
    colors = cmap(np.arange(cmap.N))
    colors[0] = [1, 1, 1, 1]
    cmap = mcolors.ListedColormap(colors)
    levels = np.linspace(-0.5, lim-0.5, lim+1)
    norm = mcolors.BoundaryNorm(boundaries=levels, ncolors=cmap.N)
    im_iso = ax.pcolormesh(iso.xc,iso.yc,iso.age*1e-3, cmap=cmap, norm=norm, shading='auto',alpha=0.5)

    # lgm
    im_lgm=ax.contour(lgm.xc, lgm.yc, lgm_leg, linestyles="--",colors='black', alpha=0.8, linewidths=1)
    
    # Legend
    gia_lab = mlines.Line2D([], [],color='red',marker='o',markersize=8,markerfacecolor='red',markeredgecolor='black',linestyle='none', label="GPS stations")
    rsl_lab = mlines.Line2D([], [],color='pink',marker='^',markersize=8,markerfacecolor='pink',markeredgecolor='black',linestyle='none',label="RSL")
    lgm_lab = mlines.Line2D([], [],color='black',linestyle="dashed", markerfacecolor='black',alpha=0.8,label="max. LGM")
    iso_lab = Rectangle((0,0), 1, 1,facecolor='gray',edgecolor='black',alpha=0.3,label='PaleoGrIS\nisochrones')
    ax.legend(handles=[lgm_lab],
                loc='lower right')
    ax.set_xlim(iso.xc.min(),iso.xc.max())
    ax.set_ylim(iso.yc.min(),iso.yc.max())
    ax.tick_params(labelbottom=False, labelleft=False)
    ax.set_aspect(1)

    return vels, bat, im_iso

In [ ]:
fig = plt.figure(figsize=(10, 7))
gs = gridspec.GridSpec(2, 3, figure=fig, 
                       height_ratios=[1, 1], 
                       width_ratios=[1, 12,12])
ax2 = fig.add_subplot(gs[0:2, 1])
vels, bat, im_iso = add_map2(ax2, topo, iso, vel)

ax = fig.add_subplot(gs[0, 2])

ax.plot(-bui["NGRIP_Age "].values/1e3,bui["NGRIP_ANN"].values-bui["NGRIP_ANN"].values[0],color="#2248B2", linewidth=1,label="annual")
ax.plot(-bui["NGRIP_Age "].values/1e3,bui["NGRIP_JJA"].values-bui["NGRIP_JJA"].values[0],color="#B22222", linewidth=1,label="summer")
ax.legend(loc='upper left', bbox_to_anchor=(0, 0.7),fontsize=10,edgecolor="none", facecolor="none")
ax.set_xlim(-22,0)
ax.set_ylim(-27,6)
ax.set_xlabel("kyr ago")
ax.set_ylabel("$\Delta \mathrm{T_{NGRIP}}$ ($^{\circ}\mathrm{C}$)")
ax.set_xticks([-20, -15, -10, -5, 0])

ax.axvspan(-22, -19, color="#deebf7", alpha=0.6)
ax.axvspan(-19, -14.8, color="white", alpha=0.8)   
ax.axvspan(-14.6, -12.8, color="#deebf7", alpha=0.9)  
ax.axvspan(-12.8, -11.7, color="#9ecae1", alpha=0.9)
ax.axvspan(-10, -6, ymin=0.15, ymax=1, color="#deebf7", alpha=0.9) 

ax.text(-20.5, -25, "LGM", ha="center", va="center")
ax.text(-13.8, -25, "BA",  ha="center", va="center")
ax.text(-12.2, -25, "YD",  ha="center", va="center")
ax.text(-7, -25, "Holocene",  ha="center", va="center")
ax.text(-8, -5, "HTM",  ha="center", va="center")

ax.plot([-12, -0.2], [-19, -19], '-*k', markersize=10, clip_on=False)
ax.text(-6, -15, "Surface elevation\nchange reconstructions", ha="center", va="center")

ax.plot([-14, -6.5], [2, 2], color="gray", linestyle="-", linewidth=1.5)
ax.plot([-14, -6.5], [2, 2], color="gray", marker="|", markersize=12, linestyle="")
ax.plot([-19, -16], [2, 2], color="gray", linestyle="-", linewidth=1.5)
ax.plot([-19, -16], [2, 2], color="gray", marker="|", markersize=12, linestyle="")
ax.text(-13, 4, "Margin reconstructions", ha="center", va="center", color="gray")

datos_tiempo = np.array(rsl.time.where(rsl.time < 20000)).flatten()/(-1000)
datos_tiempo = datos_tiempo[~np.isnan(datos_tiempo)]
datos_tiempo_reduced = datos_tiempo[::24]
y_base = -10

ax.text(-44,2,"(a)",zorder=10)
ax.text(-25,3,"(b)")
ax.text(-25,-35,"(c)")

ax = fig.add_subplot(gs[1, 2])
colors = ["#8ED7D3", "#0C2579", "#B22222", "#FC9F7D"]
labels = ["Camp Century", "NGRIP", "GRIP", "DYE-3"]
i=0
for icecsel in vinter.ice_core.values:
    vint = vinter.sel(ice_core=icecsel)
    ax.fill_between(vint.time*1e-3, (vint.z_srf-vint.z_srf[-1])*1e-3 - vint.error*1e-3, (vint.z_srf-vint.z_srf[-1])*1e-3 + vint.error*1e-3, 
                    color=colors[i], alpha=0.2)
    ax.plot(vint.time*1e-3, (vint.z_srf-vint.z_srf[-1])*1e-3, color=colors[i], label=labels[i])
    i+=1
ax.legend(loc='upper right',fontsize=10,edgecolor="none", facecolor="none")
ax.set_xlim(-11.7,0)
ax.set_xlabel("kyr ago")
ax.set_ylabel("Elevation change (km)")

ax_bar1 = fig.add_subplot(gs[0, 0])
ax_bar2 = fig.add_subplot(gs[1, 0])
ax_bar3 = ax2 #fig.add_subplot(gs[2, 1])
ax_bar1.axis('off')
ax_bar2.axis('off')
# ax_bar3.axis('off')

cbar = plt.colorbar(bat, ax=ax_bar1, fraction=0.7, pad=0.04)
cbar.set_label('Bedrock elevation (km)')
pos = cbar.ax.get_position()
cbar.ax.set_position([pos.x0, pos.y0 + 0.15, pos.width, pos.height])

cbar = plt.colorbar(vels, ax=ax_bar2, fraction=0.7, pad=0.04)
cbar.set_label('Ice surface velocity (m/yr)')
pos = cbar.ax.get_position()
cbar.ax.set_position([pos.x0, pos.y0 + 0.1, pos.width, pos.height])

cbar = plt.colorbar(im_iso, ax=ax_bar3, fraction=0.025, pad=0.02, ticks=[0,2,4,6,8,10,12,14], orientation='horizontal',
                    extend="both")
cbar.set_label('Time of last glaciation (kyr ago)')
pos = cbar.ax.get_position()

plt.subplots_adjust(left=0, right=0.99, top=0.99, bottom=0.1,wspace=0.05)
fig.savefig("../figs_final/fig1_summary_records2.pdf", dpi=300)
plt.close()


<>:16: SyntaxWarning: invalid escape sequence '\D'
<>:16: SyntaxWarning: invalid escape sequence '\D'
/tmp/ipykernel_162555/12985122.py:16: SyntaxWarning: invalid escape sequence '\D'
  ax.set_ylabel("$\Delta \mathrm{T_{NGRIP}}$ ($^{\circ}\mathrm{C}$)")
/tmp/ipykernel_162555/1384593275.py:43: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = cm.get_cmap(cmap_name, lim)
